# Convergence and stopping-criterion justification

Data: `explore_stopmode_curves.csv` (top90 runs, full trajectory). `pct_satisfiers`
marks where 50/70/90/95/100% is crossed.

**Marks:** multi-select (Ctrl/Shift) — choose which thresholds to show.
**Optimizers (Section B):** multi-select which optimizers appear as rows.
**Y axis** adjustable. **Freeze after stop**: keeps each run's last value after it stops.

The multiple thin curves in Section A are the **repetitions** (only binary fitness here).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display

plt.rcParams.update({'font.size': 14, 'axes.titlesize': 15, 'axes.labelsize': 14,
                     'legend.fontsize': 12, 'xtick.labelsize': 12, 'ytick.labelsize': 12})

OUT_DIR = 'figures_stopmode'
MARKS = [(50, '*', '#d62728', 'top50'), (70, 'D', '#2ca02c', 'top70'),
         (90, 's', '#9467bd', 'top90'),
         (95, 'P', '#17becf', 'top95'), (100, 'X', '#8c564b', 'top100')]
THRS = [m[0] for m in MARKS]
OPT_ORDER = ['umda', 'emna', 'egna', 'keda']
NET_LABEL = {'bypass': 'bypass', 'nhlv1': 'NHLy'}
METRIC_LABEL = {'acc_best':  'Best ind. rule accuracy (%)',
                'acc_mean':  'Pop. mean rule accuracy (%)',
                'acc_worst': 'Worst ind. rule accuracy (%)',
                'sat_acc_mean': 'Satisfiers mean accuracy (%)'}

def _resolve(p):
    for base in ['', '..', '../..']:
        c = os.path.join(base, p) if base else p
        if os.path.exists(c):
            return c
    return p

def load_curves():
    path = _resolve('example/explore_stopmode_curves.csv')
    for _ in range(6):
        try:
            d = pd.read_csv(path); break
        except Exception:
            d = pd.DataFrame()
    return d[d['stop_mode'] == 'top90'].copy() if len(d) else d

def opt_order(seq):
    s = set(seq)
    return [o for o in OPT_ORDER if o in s] + [o for o in sorted(s) if o not in OPT_ORDER]

CUR = load_curves()
print('rows (top90):', len(CUR))
if len(CUR):
    print('networks:', sorted(CUR.network.unique()), '| optimizers:', opt_order(CUR.optimizer.unique()),
          '| % rules:', sorted(CUR.pct.unique()))

def crossing_gen(g, thr):
    h = g[g['pct_satisfiers'] >= thr]
    return int(h['generation'].iloc[0]) if len(h) else None

def rep_mean(d, col, freeze=False):
    if freeze:
        piv = d.pivot_table(index='generation', columns='rep', values=col)
        piv = piv.reindex(range(int(piv.index.min()), int(piv.index.max()) + 1)).ffill()
        return piv.mean(axis=1)
    return d.groupby('generation')[col].mean()

def rep_mean_std(d, col, freeze=False):
    if freeze:
        piv = d.pivot_table(index='generation', columns='rep', values=col)
        piv = piv.reindex(range(int(piv.index.min()), int(piv.index.max()) + 1)).ffill()
        return piv.mean(axis=1), piv.std(axis=1).fillna(0.0)
    grp = d.groupby('generation')[col]
    return grp.mean(), grp.std().fillna(0.0)

def median_cross(d, thr):
    gens = [crossing_gen(g.sort_values('generation'), thr) for _, g in d.groupby('rep')]
    gens = [x for x in gens if x is not None]
    return int(np.median(gens)) if gens else None

def place_marks(ax, series, gx_fn, sel, legend=False):
    sel = set(sel)
    for thr, mk, c, lb in MARKS:
        if thr not in sel:
            continue
        gx = gx_fn(thr)
        if gx is not None and gx in series.index:
            ax.plot(gx, series.loc[gx], mk, color=c, ms=15 if mk == '*' else 10,
                    markeredgecolor='k', label=(lb if legend else None), zorder=5)

rows (top90): 10524
networks: ['bypass', 'nhlv1'] | optimizers: ['umda', 'emna', 'egna', 'keda'] | % rules: [5, 10, 20, 30]


In [11]:
# ====================== SECTION A — convergence (all curves) ======================
_A = {'fig': None, 'busy': 0}
a_net = W.Dropdown(description='Network')
a_opt = W.Dropdown(description='Optimizer')
a_pct = W.Dropdown(description='% rules')
a_met = W.Dropdown(description='Metric', options=['acc_best', 'acc_mean', 'sat_acc_mean'])
a_rep = W.Dropdown(description='Repetition', options=['All'])
a_marks = W.SelectMultiple(description='marks', options=THRS, value=tuple(THRS), rows=5)
a_freeze = W.Checkbox(value=True, description='freeze after stop')
a_yr = W.FloatRangeSlider(value=[0, 100], min=0, max=100, step=5, description='Y axis',
                          readout_format='.0f', continuous_update=False)
a_fname = W.Text(description='File', placeholder='auto')
a_png = W.Button(description='Export PNG', button_style='success', icon='download')
a_reload = W.Button(description='Reload', icon='refresh')
a_out = W.Output(); a_msg = W.Output()

def _a_pop(*_):
    if not len(CUR):
        return
    _A['busy'] += 1
    try:
        nets = sorted(CUR.network.unique())
        a_net.options = [(NET_LABEL.get(n, n), n) for n in nets]
        if a_net.value not in nets:
            a_net.value = 'bypass' if 'bypass' in nets else nets[0]
        sub = CUR[CUR.network == a_net.value]
        a_opt.options = opt_order(sub.optimizer.unique())
        a_pct.options = sorted(int(x) for x in sub.pct.unique())
        if a_opt.value not in a_opt.options:
            a_opt.value = 'umda' if 'umda' in a_opt.options else a_opt.options[0]
        if a_pct.value not in a_pct.options:
            a_pct.value = 10 if 10 in a_pct.options else a_pct.options[0]
        a_rep.options = ['All'] + sorted(int(r) for r in sub.rep.unique())
    finally:
        _A['busy'] -= 1

def _a_draw(*_):
    if _A['busy']:
        return
    with a_out:
        a_out.clear_output(wait=True)
        d = CUR[(CUR.network == a_net.value) & (CUR.optimizer == a_opt.value) & (CUR.pct == a_pct.value)]
        if not len(d):
            print('No data.'); return
        m = a_met.value
        fig, ax = plt.subplots(figsize=(11, 6.4))
        if a_rep.value == 'All':
            for _, g in d.groupby('rep'):
                g = g.sort_values('generation')
                ax.plot(g['generation'], g[m], color='#9bb3d4', lw=1, alpha=.8)
            mean = rep_mean(d, m, a_freeze.value)
            ax.plot(mean.index, mean.values, color='#1f4e79', lw=3, label='mean over runs')
            place_marks(ax, mean, lambda thr: median_cross(d, thr), a_marks.value, legend=True)
            sub_t = f'{d.rep.nunique()} runs + mean'
        else:
            g = d[d.rep == int(a_rep.value)].sort_values('generation').set_index('generation')[m]
            ax.plot(g.index, g.values, color='#1f4e79', lw=2.5, marker='o', ms=3, label=f'run {a_rep.value}')
            gg = d[d.rep == int(a_rep.value)].sort_values('generation')
            place_marks(ax, g, lambda thr: crossing_gen(gg, thr), a_marks.value, legend=True)
            sub_t = f'run {a_rep.value} (single)'
        ax.set_xlabel('Generation'); ax.set_ylabel(METRIC_LABEL.get(m, m))
        ax.set_ylim(a_yr.value); ax.grid(alpha=.3)
        ax.set_title(f'{NET_LABEL.get(a_net.value, a_net.value)} \u00b7 {a_opt.value.upper()} '
                     f'\u00b7 {a_pct.value}% of rules \u00b7 binary fitness \u2014 {sub_t}')
        ax.legend(loc='lower right')
        plt.tight_layout(); _A['fig'] = fig; plt.show(); plt.close(fig)

def _a_export(_):
    with a_msg:
        a_msg.clear_output(wait=True)
        if _A['fig'] is None:
            print('Draw something first.'); return
        os.makedirs(OUT_DIR, exist_ok=True)
        nm = NET_LABEL.get(a_net.value, a_net.value)
        n = a_fname.value.strip() or f'convergence_{nm}_{a_opt.value}_{a_pct.value}pct_{a_met.value}'
        n = n if n.lower().endswith('.png') else n + '.png'
        p = os.path.join(OUT_DIR, n); _A['fig'].savefig(p, dpi=150, bbox_inches='tight')
        print('Saved:', os.path.abspath(p))

def _a_reload(_):
    global CUR
    CUR = load_curves()
    with a_msg:
        a_msg.clear_output(wait=True); print(f'Reloaded: {len(CUR)} top90 rows.')
    _a_pop(); _a_draw()

a_net.observe(_a_pop, names='value')
for w in [a_net, a_opt, a_pct, a_met, a_rep, a_marks, a_freeze, a_yr]:
    w.observe(_a_draw, names='value')
a_png.on_click(_a_export); a_reload.on_click(_a_reload)
_a_pop(); _a_draw()
display(W.VBox([W.HBox([a_net, a_opt, a_pct, a_met]),
                W.HBox([a_rep, a_freeze, a_marks]),
                W.HBox([a_yr]),
                W.HBox([a_fname, a_png, a_reload]), a_msg]), a_out)

Output()

In [12]:
# ====================== SECTION B — justification grid ======================
_B = {'fig': None, 'busy': 0}
b_net = W.Dropdown(description='Network')
b_opts = W.SelectMultiple(description='optimizers', options=OPT_ORDER, value=tuple(OPT_ORDER), rows=4)
b_met = W.Dropdown(description='Metric (centre)', options=['acc_mean', 'acc_best', 'sat_acc_mean'])
b_disp = W.Dropdown(description='Spread',
                    options=['best/worst (dotted)', '\u00b11\u03c3 (band)', 'dotted + \u03c3', 'none'])
b_marks = W.SelectMultiple(description='marks', options=THRS, value=tuple(THRS), rows=5)
b_freeze = W.Checkbox(value=True, description='freeze after stop')
b_yr = W.FloatRangeSlider(value=[0, 100], min=0, max=100, step=5, description='Y axis',
                          readout_format='.0f', continuous_update=False)
b_fname = W.Text(description='File', placeholder='auto')
b_png = W.Button(description='Export PNG', button_style='success', icon='download')
b_reload = W.Button(description='Reload', icon='refresh')
b_out = W.Output(); b_msg = W.Output()

def _b_pop(*_):
    if not len(CUR):
        return
    _B['busy'] += 1
    try:
        nets = sorted(CUR.network.unique())
        b_net.options = [(NET_LABEL.get(n, n), n) for n in nets]
        if b_net.value not in nets:
            b_net.value = 'bypass' if 'bypass' in nets else nets[0]
        avail = opt_order(CUR[CUR.network == b_net.value].optimizer.unique())
        b_opts.options = avail
        b_opts.value = tuple(avail)
    finally:
        _B['busy'] -= 1

def _b_draw(*_):
    if _B['busy']:
        return
    with b_out:
        b_out.clear_output(wait=True)
        net = b_net.value; m = b_met.value; disp = b_disp.value; sel = set(b_marks.value)
        sub = CUR[CUR.network == net]
        if not len(sub):
            print('No data.'); return
        opts = [o for o in opt_order(sub.optimizer.unique()) if o in set(b_opts.value)]
        if not opts:
            print('Select at least one optimizer.'); return
        pcts = sorted(int(x) for x in sub.pct.unique())
        fig, axes = plt.subplots(len(opts), len(pcts), figsize=(4.2 * len(pcts), 3.7 * len(opts)),
                                 sharey=True, squeeze=False)
        for i, opt in enumerate(opts):
            for j, pct in enumerate(pcts):
                ax = axes[i][j]
                d = sub[(sub.optimizer == opt) & (sub.pct == pct)]
                if len(d):
                    mean, std = rep_mean_std(d, m, b_freeze.value)
                    x = mean.index.values
                    if disp in ('\u00b11\u03c3 (band)', 'dotted + \u03c3'):
                        ax.fill_between(x, mean - std, mean + std, color='#4C72B0', alpha=.18)
                    if disp in ('best/worst (dotted)', 'dotted + \u03c3'):
                        ax.plot(x, rep_mean(d, 'acc_best', b_freeze.value).values, ':', color='#55A868', lw=1.4)
                        ax.plot(x, rep_mean(d, 'acc_worst', b_freeze.value).values, ':', color='#C44E52', lw=1.4)
                    ax.plot(x, mean.values, color='#1f4e79', lw=2.2)
                    place_marks(ax, mean, lambda thr: median_cross(d, thr), sel)
                ax.set_ylim(b_yr.value); ax.grid(alpha=.3)
                if i == 0:
                    ax.set_title(f'{pct}% of rules', fontweight='bold')
                if j == 0:
                    ax.set_ylabel(f'{opt.upper()}\n\n{METRIC_LABEL.get(m, m)}')
                if i == len(opts) - 1:
                    ax.set_xlabel('Generation')
        h = [plt.Line2D([], [], color='#1f4e79', lw=2.2, label='mean over runs')]
        if disp in ('best/worst (dotted)', 'dotted + \u03c3'):
            h += [plt.Line2D([], [], color='#55A868', ls=':', lw=1.6, label='mean best individual'),
                  plt.Line2D([], [], color='#C44E52', ls=':', lw=1.6, label='mean worst individual')]
        if disp in ('\u00b11\u03c3 (band)', 'dotted + \u03c3'):
            h += [plt.Line2D([], [], color='#4C72B0', lw=6, alpha=.3, label='\u00b11\u03c3 across runs')]
        h += [plt.Line2D([], [], marker=mk, color=c, ls='', ms=10, markeredgecolor='k', label=lb)
              for thr, mk, c, lb in MARKS if thr in sel]
        fig.legend(handles=h, loc='upper center', ncol=5, frameon=False, bbox_to_anchor=(0.5, 1.06))
        fig.suptitle(f'{NET_LABEL.get(net, net)} \u2014 convergence by % of rules and optimizer (binary fitness)',
                     y=1.1, fontweight='bold')
        plt.tight_layout(); _B['fig'] = fig; plt.show(); plt.close(fig)

def _b_export(_):
    with b_msg:
        b_msg.clear_output(wait=True)
        if _B['fig'] is None:
            print('Draw something first.'); return
        os.makedirs(OUT_DIR, exist_ok=True)
        nm = NET_LABEL.get(b_net.value, b_net.value)
        n = b_fname.value.strip() or f'justification_{nm}_{b_met.value}'
        n = n if n.lower().endswith('.png') else n + '.png'
        p = os.path.join(OUT_DIR, n); _B['fig'].savefig(p, dpi=150, bbox_inches='tight')
        print('Saved:', os.path.abspath(p))

def _b_reload(_):
    global CUR
    CUR = load_curves()
    with b_msg:
        b_msg.clear_output(wait=True); print(f'Reloaded: {len(CUR)} top90 rows.')
    _b_pop(); _b_draw()

b_net.observe(_b_pop, names='value')
for w in [b_net, b_opts, b_met, b_disp, b_marks, b_freeze, b_yr]:
    w.observe(_b_draw, names='value')
b_png.on_click(_b_export); b_reload.on_click(_b_reload)
_b_pop(); _b_draw()
display(W.VBox([W.HBox([b_net, b_met, b_disp]),
                W.HBox([b_opts, b_marks]),
                W.HBox([b_freeze, b_yr]),
                W.HBox([b_fname, b_png, b_reload]), b_msg]), b_out)

Output()